# Distribution Planning Mean-Field Control Benchmark

Reference: Meunier, Pham & Reisinger, discrete-space benchmarks, Section "Distribution Planning" (`files/reference/discrete_benchmarks.tex`), based on Carmona et al.'s distribution-planning mean-field control problem.

**Model.** A population moves around the discrete torus $\mathcal X=\mathbb Z/10\mathbb Z=\{0,\ldots,9\}$, action space $\mathcal A=\{\mathrm{LEFT},\mathrm{STAY},\mathrm{RIGHT}\}$ shifting an agent by $-1,0,+1$ (mod 10). The transition kernel is **deterministic and population-independent** — unlike cybersecurity, the mean-field interaction here enters purely through the reward, which penalizes both movement and deviation from a fixed target law $\mu_\mathrm{target}=(0,0,0.05,0.10,0.20,0.30,0.20,0.10,0.05,0)$:
$$r(x,a,\mu) = -c_\mathrm{mov}|a| - \|\mu-\mu_\mathrm{target}\|_2^2, \qquad g(x,\mu) = -\|\mu-\mu_\mathrm{target}\|_2^2,$$
with $c_\mathrm{mov}=0.01$, $T=5$ for both training and validation (no train/val horizon split, unlike cybersecurity).

**Policy.** A population-dependent 2-hidden-layer MLP (width 256, $\tanh$), taking $(t,\mu)$ and outputting a $10\times3$ logit matrix, row-softmaxed — the same flat-parameter-packed pattern as cybersecurity's policy, just much larger (~76.6k parameters vs. ~1.5k), since the population state is 9-dimensional and the policy must coordinate movement across the whole state space.

As with cybersecurity, there is no known closed-form optimal policy, so this notebook has no "vs. optimal" diagnostics; instead it uses the reference's own distribution-planning-specific diagnostics: terminal/average $L_2$ mismatch to the target, the terminal transport discrepancy under the torus's cyclic metric ($W_{1,d_\mathrm{cyc}}$), and the expected cumulative movement.

In [ ]:
import sys
import time
from pathlib import Path

_notebook_start = time.perf_counter()

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
for path in (SRC, ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torch

torch.set_default_dtype(torch.float64)
torch.set_default_device("cuda" if torch.cuda.is_available() else "cpu")

from configs.distribution_planning import MID
from mfc.environments.distribution_planning import LEFT, RIGHT, STAY, DistributionPlanning, DistributionPlanningConfig
from mfc.plotting import diagnostics as viz
from scripts.train import run_all
from scripts.test import (
    average_mismatch_l2,
    cumulative_movement,
    cyclic_wasserstein,
    generalization_eval,
    gradient_diagnostics,
    load_runs,
    objective_gap,
    perturbation_coverage,
    rollout,
    state_distribution,
    terminal_mismatch_l2,
)

## Configuration and budget

This notebook demonstrates the **mid** run tier from `configs/distribution_planning.py`: one seed, the reference's horizon $T=5$, a reduced training length (5000 iterations, vs. the reference's own 100000 — kept here for wall-clock time; see `main` for the reference's full length and 5 seeds) and a reduced main batch $B=100$ (vs. the reference's $B=500$ — a memory-practicality reduction given the policy MLP's ~76.6k parameters; `main` keeps $B=500$ exactly). Run `scripts/train.py --config main` separately for the full main-tier sweep.

In [ ]:
cfg = MID
print(f"algorithms:    {cfg.algorithms}")
print(f"lambdas:       {cfg.lambdas}  (simplex perturbation scale)")
print(f"epsilon:       {cfg.epsilon}  (logit perturbation scale, fixed per context.md)")
print(f"T={cfg.horizons[0]} (training and validation)")
print(f"seeds:         {cfg.seeds}")
print(f"B={cfg.B}, n_aux={cfg.n_aux}, sigma={cfg.sigma}, lr={cfg.lr}, n_train={cfg.n_train}")
print(f"mu0 ~ Dirichlet(1,...,1) during training; validation mu0={cfg.mu0_val}")
print(f"target law: {DistributionPlanningConfig().target_law}")

## Train (or load cached results)

Loads every saved run under `runs/distribution_planning/mid/` for all three algorithms; trains first if none exist yet.

In [ ]:
env = DistributionPlanning()
runs_dir = ROOT / "runs" / "distribution_planning" / "mid"

runs = []
for alg in cfg.algorithms:
    if not list(runs_dir.glob(f"{alg}_*_seed*.pt")):
        run_all("distribution_planning", alg, "mid")
    runs += load_runs("distribution_planning", alg, "mid")

total_train_seconds = sum(r["elapsed_seconds"] for r in runs)
print(f"{len(runs)} runs loaded; total training compute time: {total_train_seconds:.1f}s ({total_train_seconds / 60:.1f} min)")
for r in sorted(runs, key=lambda r: (r["alg"], r["lam"] if r["lam"] is not None else -1)):
    tag = f"lambda={r['lam']}" if r["lam"] is not None else "(fixed epsilon)"
    print(f"  {r['alg']:<11} {tag:<16} seed={r['seed']}  elapsed={r['elapsed_seconds']:.1f}s  final validation J={r['validation_J'][-1].item():.4f}")

## Evolution of the validation reward

The exact validation objective $J_T(\theta_m;\mu_0^\mathrm{val})$ every 10 training iterations, one line per simplex $\lambda$ plus reinforce and mfreinforce.

In [ ]:
fig, ax = viz.plot_validation_curve(runs)
ax.set_title("Validation objective during training", loc="left")

## State distribution over time

The learned policy's population flow $\mu_t^\theta$ from $\mu_0^\mathrm{val}$ (the uniform law), shown for $\lambda=0.2$, against the fixed target law (dotted).

In [ ]:
by_lambda = {r["lam"]: r for r in runs if r["alg"] == "simplex"}  # mid has one seed per lambda
theta_02 = by_lambda[0.2]["theta_final"]
mu0_val = torch.tensor(cfg.mu0_val, dtype=env.dtype, device=env.device)

mu_flow = state_distribution(env, env.policy_probs, theta_02, mu0_val, cfg.horizons[0])
fig, ax = viz.plot_state_distribution(mu_flow, target_law=env.target_law, state_labels=[str(x) for x in range(env.n_states)])
ax.set_title("State distribution over time (lambda=0.2)", loc="left")

## Population-mismatch and movement diagnostics

Across the $\lambda$ sweep, at each $\lambda$'s own learned $\theta$ (reference "Evaluation criteria"):
- $\mathcal E_T^{(2)}=\|\mu_T^\theta-\mu_\mathrm{target}\|_2$ (terminal $L_2$ mismatch) and $\bar{\mathcal E}^{(2)}=\frac1{T+1}\sum_t\|\mu_t^\theta-\mu_\mathrm{target}\|_2$ (average over the episode).
- $\mathcal E_T^{(W)}=W_{1,d_\mathrm{cyc}}(\mu_T^\theta,\mu_\mathrm{target})$, the terminal transport discrepancy under the torus's cyclic metric — distinguishes a small spatial displacement from a Euclidean-comparable mismatch at distant sites.
- $\mathcal C_\mathrm{mov}=\sum_{t<T}\sum_x\mu_t(x)[1-\pi_t(\mathrm{STAY}\mid x,\mu_t)]$, the expected cumulative movement.

In [ ]:
E_T2, E_bar2, E_TW, C_mov = {}, {}, {}, {}
for lam, r in by_lambda.items():
    theta = r["theta_final"]
    flow = state_distribution(env, env.policy_probs, theta, mu0_val, cfg.horizons[0])
    E_T2[lam] = terminal_mismatch_l2(env, env.policy_probs, theta, mu0_val, cfg.horizons[0]).item()
    E_bar2[lam] = average_mismatch_l2(env, env.policy_probs, theta, mu0_val, cfg.horizons[0]).item()
    E_TW[lam] = cyclic_wasserstein(flow[-1], env.target_law).item()
    C_mov[lam] = cumulative_movement(env, env.policy_probs, theta, mu0_val, cfg.horizons[0], stay_action=STAY).item()

fig, ax = viz.plot_horizon_scaling(E_T2, xlabel="\u03bb", ylabel="mismatch", label="E_T^(2) (terminal)", integer_xaxis=False)
viz.plot_horizon_scaling(E_bar2, xlabel="\u03bb", label="E_bar^(2) (average)", color_index=1, integer_xaxis=False, ax=ax)
viz.plot_horizon_scaling(E_TW, xlabel="\u03bb", label="E_T^(W) (cyclic transport)", color_index=2, integer_xaxis=False, ax=ax)
ax.set_title("Population-mismatch diagnostics vs lambda", loc="left")

fig, ax = viz.plot_horizon_scaling(C_mov, xlabel="\u03bb", ylabel="expected cumulative movement", label="C_mov", color_index=3, integer_xaxis=False)
ax.set_title("Control effort vs lambda", loc="left")

## $J^\lambda$ vs $J$ and gradient bias/variance

As for cybersecurity, summarized as $\|\text{bias}\|$/$\|\text{std}\|$ over the full ~76.6k-dimensional MLP parameter vector (a per-component view isn't legible at this dimension).

In [ ]:
gaps, grad_diag = {}, {}
for lam, r in by_lambda.items():
    theta = r["theta_final"]
    gaps[lam] = objective_gap(env, env.policy_probs, theta, mu0_val, cfg.horizons[0], lam=lam, sigma=cfg.sigma, n_samples=2000)
    grad_diag[lam] = gradient_diagnostics(
        env, env.policy_probs, theta, mu0_val, cfg.horizons[0],
        lam=lam, n_aux=cfg.n_aux, B=cfg.B, sigma=cfg.sigma, reps=10,
    )

fig, ax = viz.plot_objective_gap(gaps)
ax.set_title("J vs J^lambda", loc="left")

bias_norm = {lam: grad_diag[lam]["bias"].norm().item() for lam in grad_diag}
std_norm = {lam: grad_diag[lam]["std"].norm().item() for lam in grad_diag}
fig, ax = viz.plot_horizon_scaling(bias_norm, xlabel="\u03bb", ylabel="norm over ~76.6k MLP parameters", label="||bias||", integer_xaxis=False)
viz.plot_horizon_scaling(std_norm, xlabel="\u03bb", label="||std||", color_index=1, integer_xaxis=False, ax=ax)
ax.set_title("Simplex gradient estimator: bias/std norms vs lambda", loc="left")

## Simplex-perturbation coverage: $d_{TV}(M^\lambda,\mu)\le\lambda$

Checked (as elsewhere) by direct sampling at a few representative population laws, now $N=10$-dimensional: the validation law, the target law itself, and a point from the learned flow.

In [ ]:
coverage = perturbation_coverage(torch.stack([mu0_val, env.target_law, mu_flow[2]]), lam=0.2, sigma=cfg.sigma, n_samples=5000)
fig, ax = viz.plot_perturbation_coverage(coverage, lam=0.2, mu_labels=["mu0_val (uniform)", "target_law", "mu_flow[2]"])
ax.set_title("d_TV(M^lambda, mu) vs the lambda=0.2 bound", loc="left")

for r in coverage:
    assert r["within_bound"], "the perturbation theorem's bound should never be violated"
print("bound holds for all sampled mu (as guaranteed by the theorem).")

## Initial, target, and terminal distributions

The reference's own qualitative check (reference "Evaluation criteria"): does a high validation value come from genuine transport toward the target, or from excessive local oscillation?

In [ ]:
fig, ax = viz.plot_distribution_comparison(
    {"initial (mu0_val)": mu0_val, "target": env.target_law, "terminal (learned)": mu_flow[-1]},
    state_labels=[str(x) for x in range(env.n_states)],
)
ax.set_title("Initial vs target vs terminal distribution (lambda=0.2)", loc="left")

## Sample trajectory under the learned policy

One sampled state trajectory ($\lambda=0.2$) from $\mu_0^\mathrm{val}$, over $T=5$ steps on the torus.

In [ ]:
learned_traj = rollout(env, env.policy_probs, theta_02, mu0_val, T=cfg.horizons[0], generator=torch.Generator(device=env.device).manual_seed(0))
fig, ax = viz.plot_trajectories(learned_traj)
ax.set_title("Sample trajectory (lambda=0.2)", loc="left")

## Generalization without retraining

Evaluating the $\lambda=0.2$ learned $\theta$ exactly (no retraining) under different initial laws, a longer horizon, and a stronger/weaker movement penalty.

In [ ]:
corner = torch.zeros(env.n_states, dtype=env.dtype, device=env.device)
corner[0] = 1.0
scenarios = [
    {"name": "baseline (mu0_val)"},
    {"name": "mu0=all mass at 0", "mu0": corner},
    {"name": "mu0=target_law", "mu0": env.target_law},
    {"name": "T=10", "T": 10},
    {"name": "2x movement penalty", "env": DistributionPlanning(DistributionPlanningConfig(c_mov=0.02))},
    {"name": "no movement penalty", "env": DistributionPlanning(DistributionPlanningConfig(c_mov=0.0))},
]
gen_results = generalization_eval(env, env.policy_probs, theta_02, mu0_val, cfg.horizons[0], scenarios)
fig, ax = viz.plot_generalization(gen_results)
ax.set_title("J under different scenarios (theta fixed, no retraining)", loc="left")

## Comparing the three algorithms

At `mid`'s single seed, final validation objective for each algorithm — simplex at its best-performing $\lambda$, reinforce (which omits the population-sensitivity correction $Q_t(D_t)$), and mfreinforce. A single seed can't separate genuine differences from noise; `main`'s 5 seeds are needed for a real comparison.

In [ ]:
by_alg = {}
for r in runs:
    by_alg.setdefault(r["alg"], []).append(r)

best_simplex = max(by_alg["simplex"], key=lambda r: r["validation_J"][-1].item())
print(f"best simplex lambda: {best_simplex['lam']}")
for alg in cfg.algorithms:
    r = best_simplex if alg == "simplex" else by_alg[alg][0]
    print(f"  {alg:<11} final validation J = {r['validation_J'][-1].item():.4f}")

## Summary

Total notebook runtime (including any training performed in this run):

In [ ]:
print(f"total notebook runtime: {time.perf_counter() - _notebook_start:.1f}s")